# AI Product Image Generator with Gradio


Build an AI powered product image generation system using:
1. An LLM for prompt engineering
2. Stable Diffusion for image generation
3. Gradio for the web interface

**Enable GPU before running:** Settings → Accelerator → GPU T4 x2

---
## Step 0: Setup

In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected! Enable GPU in notebook settings.")

In [ ]:
# Install required libraries
!pip install -q transformers diffusers accelerate gradio sentencepiece
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from diffusers import StableDiffusionPipeline
import gradio as gr
from PIL import Image

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Create output directories
import os
os.makedirs("images", exist_ok=True)
print("Output directories ready.")

---
## Step 1: Load the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/abhinayvanjari/product-descriptions/product_descriptions.csv")
print(f"Loaded {len(df)} product descriptions")
df.head()

---
## Step 2: Load LLM for Prompt Engineering

Load an LLM and write a function that takes a simple product description and returns a detailed, creative prompt for image generation.

**You can use any open source LLM.** Example: Qwen 2.5-0.5B-Instruct

In [ ]:
# ============================================================
# TODO: Load your LLM model and tokenizer here
# ============================================================
# Example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

QWEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL, torch_dtype=torch.float16, device_map="auto"
)



In [ ]:
# ============================================================
# TODO: Write your prompt engineering function
# ============================================================
# This function should:
# 1. Take a simple product description as input
# 2. Use the LLM to rewrite it into a detailed image generation prompt
# 3. Return the engineered prompt as a string
#
# Tips:
# - Use a good system prompt (role + context + instruction)
# - Keep output under 60 words (CLIP token limit is 77)
# - Include details about lighting, style, composition
def engineer_prompt(product_description):
    """
    Optimized for Prompt Quality Score.
    Uses a strict system prompt to ensure professional e-commerce aesthetics.
    """
    system_instruction = (
        "You are an expert AI prompt engineer for e-commerce product photography. "
        "Rewrite the user's product description into a highly detailed, vivid, and professional image prompt. "
        "Include details about studio lighting (e.g., softbox, rim lighting), professional composition (e.g., macro shot, centered), "
        "and high-end textures. Keep the output under 65 words. Output ONLY the improved prompt."
    )
    
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Product: {product_description}"}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    generated_ids = llm_model.generate(model_inputs.input_ids, max_new_tokens=100, temperature=0.7)
    response = tokenizer.batch_decode(generated_ids[:, model_inputs.input_ids.shape[-1]:], skip_special_tokens=True)[0]
    
    return response.strip()

# Test it
test = engineer_prompt("red wireless headphones on a white background")
print(test)

---
## Step 3: Load Stable Diffusion for Image Generation

Load Stable Diffusion and write a function that generates an image from a prompt.

In [ ]:
# ============================================================
# TODO: Load Stable Diffusion pipeline here
# ============================================================
# Example:
# from diffusers import StableDiffusionPipeline

# YOUR CODE HERE
SD_MODEL = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL, 
    torch_dtype=torch.float16, 
    safety_checker=None  # Speed up generation
)
pipe = pipe.to(device)


In [ ]:
# ============================================================
# TODO: Write your image generation function
# ============================================================
# This function should:
# 1. Take an engineered prompt as input
# 2. Generate an image using Stable Diffusion
# 3. Return a PIL Image
def generate_image(engineered_prompt):
    # Standard e-commerce parameters for consistency
    image = pipe(
        prompt=engineered_prompt,
        negative_prompt="blurry, low quality, distorted, text, watermark, messy background, dark shadows",
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]
    return image

# Test it
img = generate_image("A professional photo of red headphones on white background")
display(img)

---
## Step 4: Build Gradio Interface

Create a Gradio UI that connects prompt engineering and image generation.

**Your interface must:**
- Accept a product description as input
- Display the engineered prompt
- Display the generated image

In [ ]:
# ============================================================
# TODO: Build your Gradio interface here
# ============================================================
# Example: import gradio as gr
# Use gr.Blocks or gr.Interface



# YOUR CODE HERE
def master_workflow(product_desc):
    eng_prompt = engineer_prompt(product_desc)
    img = generate_image(eng_prompt)
    return eng_prompt, img

with gr.Blocks(title="AI Product Photographer") as demo:
    gr.Markdown("# 🛍️ AI Product Image System")
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Enter Simple Product Description", placeholder="e.g., a leather watch")
            btn = gr.Button("Generate Professional Image", variant="primary")
        with gr.Column():
            output_prompt = gr.Textbox(label="Engineered Prompt (LLM)")
            output_image = gr.Image(label="Generated Product Photo")
    
    btn.click(fn=master_workflow, inputs=input_text, outputs=[output_prompt, output_image])

# Use inline=True for Kaggle/Colab notebooks
demo.launch(share=True, inline=True)


---
## Step 5: Generate Outputs for All 15 Products

Run your pipeline on all product descriptions and save the results.

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/abhinayvanjari/product-descriptions/product_descriptions.csv")
print(df.columns.tolist())

In [ ]:
import os
import pandas as pd

# 1. SETUP: Paths
INPUT_CSV = "/kaggle/input/datasets/abhinayvanjari/product-descriptions/product_descriptions.csv"
output_dir = "/kaggle/working/images"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. LOAD DATA
df = pd.read_csv(INPUT_CSV)

# --- SMART COLUMN SELECTOR ---
# We look for a column that contains strings (text), not just numbers.
text_col = None
for col in df.columns:
    # If the first non-null value in this column is a string, we use it
    if df[col].dtype == 'object':
        text_col = col
        break

if text_col is None:
    # If no text column is found, we'll try the second column (index 1)
    text_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]

product_list = df[text_col].tolist()
print(f"✅ Using text column: '{text_col}'")
print(f"✅ Found {len(product_list)} products to generate.")

# 3. GENERATION LOOP
results = []
for i, description in enumerate(product_list):
    product_id = i + 1 
    # Safety check: ensure description is a string
    desc_str = str(description)
    print(f"📸 Processing {product_id}/15: {desc_str[:30]}...")
    
    try:
        # Generate prompt and image
        eng_prompt = engineer_prompt(desc_str)
        img = generate_image(eng_prompt)
        
        # Save image as img_1.png
        image_filename = f"img_{product_id}.png"
        image_path = os.path.join(output_dir, image_filename)
        img.save(image_path)
        
        results.append({
            "id": product_id,
            "description": desc_str, 
            "engineered_prompt": eng_prompt
        })
    except Exception as e:
        print(f"⚠️ Error on product {product_id}: {e}")
        results.append({
            "id": product_id, 
            "description": desc_str,
            "engineered_prompt": "High-end professional product photography"
        })

# 4. SAVE CSV
submission_df = pd.DataFrame(results)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print("\n" + "="*30)
print(f"✅ Created submission.csv with {len(submission_df)} rows.")
print(f"✅ Saved {len(os.listdir(output_dir))} images to {output_dir}")
print("="*30)

---
## Step 6: Evaluation (Do Not Modify This Section)

Run all the cells below to compute your scores and generate `final_scores.csv` for leaderboard submission.

**DO NOT MODIFY THESE CELLS.**

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgMTogUHJvbXB0IFF1YWxpdHkgKDQwJSB3ZWlnaHQpCiMgRE8gTk9UIE1PRElGWQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgpmcm9tIHNrbGVhcm4ubWV0cmljcy5wYWlyd2lzZSBpbXBvcnQgY29zaW5lX3NpbWlsYXJpdHkKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCnByaW50KCI9IiAqIDUwKQpwcmludCgiICBFVkFMVUFUSU5HIFBST01QVCBRVUFMSVRZIikKcHJpbnQoIj0iICogNTApCgpldmFsX21vZGVsID0gU2VudGVuY2VUcmFuc2Zvcm1lcigiYWxsLU1pbmlMTS1MNi12MiIpCgpzdWJtaXNzaW9uID0gcGQucmVhZF9jc3YoInN1Ym1pc3Npb24uY3N2IikKZ29sZCA9IHBkLnJlYWRfY3N2KCJodHRwczovL3MzLmFwLXNvdXRoLTEuYW1hem9uYXdzLmNvbS9uZXctYXNzZXRzLmNjYnAuaW4vZnJvbnRlbmQvY29udGVudC9haW1sL01hc3RlcmNsYXNzX05JQVQvZ29sZF9zdGFuZGFyZF9wcm9tcHRzLmNzdiIpCgptZXJnZWQgPSBzdWJtaXNzaW9uLm1lcmdlKGdvbGQsIG9uPSJpZCIpCgpwcm9tcHRfc2NvcmVzID0ge30KZm9yIF8sIHJvdyBpbiBtZXJnZWQuaXRlcnJvd3MoKToKICAgIHN0dWRlbnRfZW1iID0gZXZhbF9tb2RlbC5lbmNvZGUoW3N0cihyb3dbImVuZ2luZWVyZWRfcHJvbXB0Il0pXSkKICAgIGdvbGRfZW1iID0gZXZhbF9tb2RlbC5lbmNvZGUoW3N0cihyb3dbImdvbGRfcHJvbXB0Il0pXSkKICAgIHNpbSA9IGZsb2F0KGNvc2luZV9zaW1pbGFyaXR5KHN0dWRlbnRfZW1iLCBnb2xkX2VtYilbMF1bMF0pCiAgICBzaW0gPSBtYXgoMC4wLCBtaW4oMS4wLCBzaW0pKQogICAgcHJvbXB0X3Njb3Jlc1tyb3dbImlkIl1dID0gcm91bmQoc2ltLCA0KQogICAgcHJpbnQoZiIgIFByb2R1Y3Qge3Jvd1snaWQnXToyZH06IHByb21wdF9zY29yZSA9IHtzaW06LjRmfSIpCgphdmdfcHJvbXB0ID0gbnAubWVhbihsaXN0KHByb21wdF9zY29yZXMudmFsdWVzKCkpKQpwcmludChmIlxuICBBdmVyYWdlIFByb21wdCBTY29yZToge2F2Z19wcm9tcHQ6LjRmfSIpCnByaW50KCI9IiAqIDUwKQ=="  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgMjogSW1hZ2UgUXVhbGl0eSAoNDAlIHdlaWdodCkKIyBETyBOT1QgTU9ESUZZCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CmltcG9ydCB0b3JjaApmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQ0xJUFByb2Nlc3NvciwgQ0xJUE1vZGVsCmZyb20gUElMIGltcG9ydCBJbWFnZQppbXBvcnQgbnVtcHkgYXMgbnAKCnByaW50KCI9IiAqIDUwKQpwcmludCgiICBFVkFMVUFUSU5HIElNQUdFIFFVQUxJVFkiKQpwcmludCgiPSIgKiA1MCkKCmNsaXBfbW9kZWwgPSBDTElQTW9kZWwuZnJvbV9wcmV0cmFpbmVkKCJvcGVuYWkvY2xpcC12aXQtYmFzZS1wYXRjaDMyIikKY2xpcF9wcm9jZXNzb3IgPSBDTElQUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZCgib3BlbmFpL2NsaXAtdml0LWJhc2UtcGF0Y2gzMiIpCgpkZl9ldmFsID0gcGQucmVhZF9jc3YoImh0dHBzOi8vczMuYXAtc291dGgtMS5hbWF6b25hd3MuY29tL25ldy1hc3NldHMuY2NicC5pbi9mcm9udGVuZC9jb250ZW50L2FpbWwvTWFzdGVyY2xhc3NfTklBVC9wcm9kdWN0X2Rlc2NyaXB0aW9ucy5jc3YiKQoKaW1hZ2Vfc2NvcmVzID0ge30KZm9yIF8sIHJvdyBpbiBkZl9ldmFsLml0ZXJyb3dzKCk6CiAgICBwaWQgPSByb3dbImlkIl0KICAgIGRlc2MgPSByb3dbInByb2R1Y3RfZGVzY3JpcHRpb24iXQogICAgaW1nX3BhdGggPSBmImltYWdlcy9pbWdfe3BpZH0ucG5nIgogICAgCiAgICB0cnk6CiAgICAgICAgaW1hZ2UgPSBJbWFnZS5vcGVuKGltZ19wYXRoKS5jb252ZXJ0KCJSR0IiKQogICAgICAgIGlucHV0cyA9IGNsaXBfcHJvY2Vzc29yKHRleHQ9W2Rlc2NdLCBpbWFnZXM9aW1hZ2UsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSkKICAgICAgICAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0cHV0cyA9IGNsaXBfbW9kZWwoKippbnB1dHMpCiAgICAgICAgICAgIHJhd19zY29yZSA9IG91dHB1dHMubG9naXRzX3Blcl9pbWFnZS5pdGVtKCkKICAgICAgICAKICAgICAgICBub3JtYWxpemVkID0gZmxvYXQobnAuY2xpcCgocmF3X3Njb3JlIC0gMTUuMCkgLyAyMC4wLCAwLjAsIDEuMCkpCiAgICAgICAgaW1hZ2Vfc2NvcmVzW3BpZF0gPSByb3VuZChub3JtYWxpemVkLCA0KQogICAgICAgIHByaW50KGYiICBQcm9kdWN0IHtwaWQ6MmR9OiBDTElQID0ge3Jhd19zY29yZTouMmZ9LCBpbWFnZV9zY29yZSA9IHtub3JtYWxpemVkOi40Zn0iKQogICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgIHByaW50KGYiICBQcm9kdWN0IHtwaWQ6MmR9OiBJTUFHRSBOT1QgRk9VTkQgKHNjb3JlID0gMCkiKQogICAgICAgIGltYWdlX3Njb3Jlc1twaWRdID0gMC4wCgphdmdfaW1hZ2UgPSBucC5tZWFuKGxpc3QoaW1hZ2Vfc2NvcmVzLnZhbHVlcygpKSkKcHJpbnQoZiJcbiAgQXZlcmFnZSBJbWFnZSBTY29yZToge2F2Z19pbWFnZTouNGZ9IikKcHJpbnQoIj0iICogNTAp"  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgMzogR3JhZGlvIFVJIENoZWNrCiMgRE8gTk9UIE1PRElGWQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpwcmludCgiPSIgKiA1MCkKcHJpbnQoIiAgRVZBTFVBVElORyBHUkFESU8gVUkiKQpwcmludCgiPSIgKiA1MCkKCmdyYWRpb192YWwgPSAwLjAKCnRyeToKICAgIGltcG9ydCBncmFkaW8gYXMgZ3IKICAgIAogICAgZ3JhZGlvX2ZvdW5kID0gRmFsc2UKICAgIGZvciBrLCB2IGluIGxpc3QoZ2xvYmFscygpLml0ZW1zKCkpOgogICAgICAgIGlmIGsuc3RhcnRzd2l0aCgiXyIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2LCAoZ3IuQmxvY2tzLCkpOgogICAgICAgICAgICAgICAgZ3JhZGlvX2ZvdW5kID0gVHJ1ZQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBleGNlcHQ6CiAgICAgICAgICAgIHBhc3MKICAgIAogICAgaWYgZ3JhZGlvX2ZvdW5kOgogICAgICAgIGdyYWRpb192YWwgPSAxLjAKICAgICAgICBwcmludCgiICBHcmFkaW8gaW50ZXJmYWNlIERFVEVDVEVEIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIiAgR3JhZGlvIGludGVyZmFjZSBOT1QgRk9VTkQiKQogICAgICAgIHByaW50KCIgIE1ha2Ugc3VyZSB5b3UgY3JlYXRlIGEgZ3IuQmxvY2tzKCkgb3IgZ3IuSW50ZXJmYWNlKCkgb2JqZWN0IikKZXhjZXB0IEltcG9ydEVycm9yOgogICAgcHJpbnQoIiAgR3JhZGlvIGlzIG5vdCBpbnN0YWxsZWQiKQoKcHJpbnQoZiIgIEdyYWRpbyBTY29yZToge2dyYWRpb192YWx9IikKcHJpbnQoIj0iICogNTAp"  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgNDogR2VuZXJhdGUgZmluYWxfc2NvcmVzLmNzdiBmb3IgbGVhZGVyYm9hcmQKIyBETyBOT1QgTU9ESUZZCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgQ29tYmluZWQgc2NvcmUgcGVyIHByb2R1Y3Q6CiMgc2NvcmUgPSBwcm9tcHRfc2NvcmUgKiAwLjQgKyBpbWFnZV9zY29yZSAqIDAuNCArIGdyYWRpb19zY29yZSAqIDAuMgojIFBlcmZlY3Qgc2NvcmUgPSAxLjAgcGVyIHByb2R1Y3QuIExlYWRlcmJvYXJkIHVzZXMgUk1TRSAobG93ZXIgPSBiZXR0ZXIpLgoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCnJvd3MgPSBbXQpmb3IgcGlkIGluIHJhbmdlKDEsIDE2KToKICAgIHAgPSBwcm9tcHRfc2NvcmVzLmdldChwaWQsIDAuMCkKICAgIGkgPSBpbWFnZV9zY29yZXMuZ2V0KHBpZCwgMC4wKQogICAgZyA9IGdyYWRpb192YWwKICAgIGNvbWJpbmVkID0gcm91bmQocCAqIDAuNCArIGkgKiAwLjQgKyBnICogMC4yLCA0KQogICAgcm93cy5hcHBlbmQoeyJpZCI6IHBpZCwgInNjb3JlIjogY29tYmluZWR9KQoKZmluYWwgPSBwZC5EYXRhRnJhbWUocm93cykKZmluYWwudG9fY3N2KCJmaW5hbF9zY29yZXMuY3N2IiwgaW5kZXg9RmFsc2UpCgpwcmludCgiPSIgKiA1MCkKcHJpbnQoIiAgICAgICAgIEZJTkFMIEVWQUxVQVRJT04gUkVTVUxUUyIpCnByaW50KCI9IiAqIDUwKQpwcmludCgpCmZvciBfLCByIGluIGZpbmFsLml0ZXJyb3dzKCk6CiAgICBwcmludChmIiAgUHJvZHVjdCB7aW50KHJbJ2lkJ10pOjJkfTogc2NvcmUgPSB7clsnc2NvcmUnXTouNGZ9IikKcHJpbnQoKQoKYXZnX3Byb21wdCA9IG5wLm1lYW4obGlzdChwcm9tcHRfc2NvcmVzLnZhbHVlcygpKSkKYXZnX2ltYWdlID0gbnAubWVhbihsaXN0KGltYWdlX3Njb3Jlcy52YWx1ZXMoKSkpCgpwcmludChmIiAgUHJvbXB0IFF1YWxpdHkgKGF2ZykgOiB7YXZnX3Byb21wdDouNGZ9IikKcHJpbnQoZiIgIEltYWdlIFF1YWxpdHkgIChhdmcpIDoge2F2Z19pbWFnZTouNGZ9IikKcHJpbnQoZiIgIEdyYWRpbyBVSSAgICAgICAgICAgIDoge2dyYWRpb192YWw6LjFmfSIpCnByaW50KCkKcHJpbnQoZiIgIFByb21wdCBQb2ludHMgIDoge2F2Z19wcm9tcHQgKiA0MDo1LjFmfSAvIDQwIikKcHJpbnQoZiIgIEltYWdlIFBvaW50cyAgIDoge2F2Z19pbWFnZSAqIDQwOjUuMWZ9IC8gNDAiKQpwcmludChmIiAgR3JhZGlvIFBvaW50cyAgOiB7Z3JhZGlvX3ZhbCAqIDIwOjUuMWZ9IC8gMjAiKQp0b3RhbCA9IGF2Z19wcm9tcHQgKiA0MCArIGF2Z19pbWFnZSAqIDQwICsgZ3JhZGlvX3ZhbCAqIDIwCnByaW50KGYiICB7J+KUgCcgKiAzMH0iKQpwcmludChmIiAgVE9UQUwgU0NPUkUgICAgOiB7dG90YWw6NS4xZn0gLyAxMDAiKQpwcmludCgpCnByaW50KCI9IiAqIDUwKQpwcmludCgiICBmaW5hbF9zY29yZXMuY3N2IHNhdmVkLiIpCnByaW50KCIgIFN1Ym1pdCB0aGlzIGZpbGUgdG8gdGhlIGNvbXBldGl0aW9uIGxlYWRlcmJvYXJkLiIpCnByaW50KCI9IiAqIDUwKQ=="  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))